# 04 — Customer Lifetime Value Prediction

## Purpose

This notebook builds the first supervised machine-learning model for the e-commerce intelligence platform.

We use historical customer behavior to predict a **future customer revenue target** rather than predicting lifetime value from the same transactions used to create the features.

### Modeling principle

```text
Historical observation period
        ↓
Customer features
        ↓
Future revenue target
        ↓
XGBoost regression
        ↓
CLV-style revenue prediction
```

This temporal setup is important because using the customer's full historical revenue as both an input and target would create target leakage.

## Business questions

- Which customers are expected to generate the most future revenue?
- How accurately can future customer value be estimated?
- Which customer behaviors are most predictive of future value?
- Which customers have high predicted value but low recent activity?
- Can predicted value support retention prioritization?

The model is a **future revenue prediction model** and should be described honestly as such in the portfolio.

In [ ]:
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "customer_features.csv"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature file: {DATA_PATH}")

## 1. Load customer features

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. "
        "Run src/feature_engineering.py first."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())

required_columns = [
    "customer_unique_id",
    "first_purchase_date",
    "last_purchase_date",
    "order_count",
    "total_revenue",
]

missing = [
    column for column in required_columns
    if column not in df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

## 2. Define the temporal modeling framework

The feature table contains customer behavior aggregated across the full observed dataset.

For a genuine future-value model, we need an observation cutoff.

We therefore rebuild the customer features from order-level data using:

```text
80% historical time → feature/observation period
20% future time      → target period
```

This prevents future purchases from entering the predictors.

In [ ]:
from sqlalchemy import text
from src.database import get_engine

engine = get_engine()

cutoff_query = text("""
SELECT PERCENTILE_CONT(0.80)
    WITHIN GROUP (ORDER BY order_purchase_timestamp)
    AS cutoff_timestamp
FROM olist_orders
""")

with engine.connect() as connection:
    cutoff_df = pd.read_sql(cutoff_query, connection)

cutoff_timestamp = pd.to_datetime(
    cutoff_df.loc[0, "cutoff_timestamp"]
)

print("Observation cutoff:", cutoff_timestamp)

## 3. Build historical customer features and future revenue

In [ ]:
historical_query = text("""
WITH historical_orders AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp
    FROM olist_customers c
    JOIN olist_orders o
        ON c.customer_id = o.customer_id
    WHERE o.order_purchase_timestamp <= :cutoff
),
historical_items AS (
    SELECT
        oi.order_id,
        SUM(oi.price) AS product_revenue,
        SUM(oi.freight_value) AS freight_revenue,
        COUNT(*) AS item_count,
        COUNT(DISTINCT oi.product_id) AS unique_products
    FROM olist_order_items oi
    GROUP BY oi.order_id
),
future_revenue AS (
    SELECT
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value) AS future_revenue
    FROM olist_customers c
    JOIN olist_orders o
        ON c.customer_id = o.customer_id
    JOIN olist_order_items oi
        ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp > :cutoff
    GROUP BY c.customer_unique_id
)
SELECT
    ho.customer_unique_id,
    COUNT(DISTINCT ho.order_id) AS historical_order_count,
    SUM(COALESCE(hi.product_revenue, 0)
        + COALESCE(hi.freight_revenue, 0)) AS historical_revenue,
    SUM(COALESCE(hi.item_count, 0)) AS historical_item_count,
    SUM(COALESCE(hi.unique_products, 0)) AS historical_unique_products,
    MIN(ho.order_purchase_timestamp) AS first_purchase_date,
    MAX(ho.order_purchase_timestamp) AS last_purchase_date,
    COALESCE(fr.future_revenue, 0) AS future_revenue
FROM historical_orders ho
LEFT JOIN historical_items hi
    ON ho.order_id = hi.order_id
LEFT JOIN future_revenue fr
    ON ho.customer_unique_id = fr.customer_unique_id
GROUP BY
    ho.customer_unique_id,
    fr.future_revenue
""")

with engine.connect() as connection:
    modeling_df = pd.read_sql(
        historical_query,
        connection,
        params={"cutoff": cutoff_timestamp},
    )

print("Modeling dataset shape:", modeling_df.shape)
display(modeling_df.head())

## 4. Add historical behavioral features

We derive features using only the historical period:

- historical recency
- historical frequency
- historical monetary value
- average historical order value
- average items per order
- customer observation age

The target remains:

```text
future_revenue
```

In [ ]:
date_columns = [
    "first_purchase_date",
    "last_purchase_date",
]

for column in date_columns:
    modeling_df[column] = pd.to_datetime(
        modeling_df[column],
        errors="coerce",
    )

modeling_df["historical_recency_days"] = (
    cutoff_timestamp
    - modeling_df["last_purchase_date"]
).dt.total_seconds() / 86400.0

modeling_df["historical_customer_age_days"] = (
    modeling_df["last_purchase_date"]
    - modeling_df["first_purchase_date"]
).dt.total_seconds() / 86400.0

modeling_df["historical_customer_age_days"] = (
    modeling_df["historical_customer_age_days"].clip(lower=0)
)

modeling_df["historical_average_order_value"] = (
    modeling_df["historical_revenue"]
    / modeling_df["historical_order_count"].replace(0, np.nan)
)

modeling_df["historical_average_items_per_order"] = (
    modeling_df["historical_item_count"]
    / modeling_df["historical_order_count"].replace(0, np.nan)
)

modeling_df["historical_orders_per_active_day"] = (
    modeling_df["historical_order_count"]
    / modeling_df["historical_customer_age_days"].replace(0, np.nan)
)

ratio_columns = [
    "historical_average_order_value",
    "historical_average_items_per_order",
    "historical_orders_per_active_day",
]

modeling_df[ratio_columns] = (
    modeling_df[ratio_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

modeling_df["future_revenue"] = pd.to_numeric(
    modeling_df["future_revenue"],
    errors="coerce",
).fillna(0)

display(modeling_df.describe().T)

## 5. Inspect the target

Future revenue is usually strongly right-skewed.

We inspect both the raw and log-transformed target.

The model will use:

```text
log1p(future_revenue)
```

internally and transform predictions back to the original currency scale.

In [ ]:
fig = px.histogram(
    modeling_df,
    x="future_revenue",
    nbins=60,
    title="Future Customer Revenue Distribution",
    labels={"future_revenue": "Future Revenue"},
)
fig.show()

target_log = np.log1p(
    modeling_df["future_revenue"].clip(lower=0)
)

fig = px.histogram(
    x=target_log,
    nbins=60,
    title="Log-Transformed Future Revenue",
    labels={"x": "log(1 + future revenue)"},
)
fig.show()

print(
    "Customers with zero future revenue:",
    int((modeling_df["future_revenue"] == 0).sum()),
)

print(
    "Customers with future revenue:",
    int((modeling_df["future_revenue"] > 0).sum()),
)

## 6. Define the modeling matrix

Only historical information is used.

We deliberately exclude:

- `future_revenue`
- customer ID
- raw dates
- current full-period features from `customer_features.csv`

This prevents future information from leaking into the model.

In [ ]:
feature_columns = [
    "historical_recency_days",
    "historical_order_count",
    "historical_revenue",
    "historical_item_count",
    "historical_unique_products",
    "historical_average_order_value",
    "historical_average_items_per_order",
    "historical_orders_per_active_day",
]

feature_columns = [
    column
    for column in feature_columns
    if column in modeling_df.columns
]

X = modeling_df[feature_columns].copy()
y = modeling_df["future_revenue"].copy()

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

print("Features:")
for column in feature_columns:
    print(" -", column)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

## 7. Train/test split

Because the feature table already uses an earlier time period and the target uses future revenue, the most important temporal leakage issue has been handled.

We now use a deterministic customer-level train/test split for model validation.

A fixed random seed makes the experiment reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training target mean:", round(y_train.mean(), 2))
print("Testing target mean:", round(y_test.mean(), 2))

## 8. Train XGBoost

XGBoost is used because it handles:

- nonlinear relationships,
- feature interactions,
- mixed feature scales,
- long-tailed business behavior,
- tabular data effectively.

The target is modeled in log space using `TransformedTargetRegressor`.

In [ ]:
base_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)

model = TransformedTargetRegressor(
    regressor=base_model,
    func=np.log1p,
    inverse_func=np.expm1,
)

model.fit(X_train, y_train)

print("XGBoost CLV-style revenue model trained.")

## 9. Evaluate the model

In [ ]:
predictions = model.predict(X_test)

predictions = np.clip(
    predictions,
    a_min=0,
    a_max=None,
)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

metrics = pd.DataFrame({
    "metric": [
        "MAE",
        "RMSE",
        "R2",
    ],
    "value": [
        mae,
        rmse,
        r2,
    ],
})

display(metrics.round(4))

## 10. Actual vs predicted future revenue

In [ ]:
prediction_df = pd.DataFrame({
    "actual_future_revenue": y_test.values,
    "predicted_future_revenue": predictions,
})

fig = px.scatter(
    prediction_df,
    x="actual_future_revenue",
    y="predicted_future_revenue",
    opacity=0.55,
    title="Actual vs Predicted Future Revenue",
    labels={
        "actual_future_revenue": "Actual Future Revenue",
        "predicted_future_revenue": "Predicted Future Revenue",
    },
)

max_value = max(
    prediction_df["actual_future_revenue"].max(),
    prediction_df["predicted_future_revenue"].max(),
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_value,
    y1=max_value,
)

fig.show()

## 11. Prediction error analysis

In [ ]:
prediction_df["error"] = (
    prediction_df["predicted_future_revenue"]
    - prediction_df["actual_future_revenue"]
)

prediction_df["absolute_error"] = (
    prediction_df["error"].abs()
)

prediction_df["absolute_percentage_error"] = (
    prediction_df["absolute_error"]
    / prediction_df["actual_future_revenue"].replace(0, np.nan)
    * 100
)

display(
    prediction_df[
        [
            "actual_future_revenue",
            "predicted_future_revenue",
            "error",
            "absolute_error",
        ]
    ]
    .describe()
    .T
)

fig = px.histogram(
    prediction_df,
    x="error",
    nbins=60,
    title="Prediction Error Distribution",
    labels={"error": "Predicted − Actual"},
)
fig.show()

## 12. Feature importance

XGBoost's native feature importance provides a first interpretation of which historical behaviors contribute most strongly to the model.

This is not causal analysis.

Later we will add SHAP for customer-level explanations.

In [ ]:
xgb_model = model.regressor_

importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": xgb_model.feature_importances_,
}).sort_values(
    "importance",
    ascending=False,
)

display(importance)

fig = px.bar(
    importance.sort_values("importance"),
    x="importance",
    y="feature",
    orientation="h",
    title="XGBoost Feature Importance",
    labels={
        "importance": "Importance",
        "feature": "Feature",
    },
)
fig.show()

## 13. Generate customer-level predicted value

We now score every customer represented in the modeling dataset.

The resulting table contains:

- historical behavior,
- future actual revenue,
- predicted future revenue,
- prediction error.

The actual future revenue column is useful for offline evaluation and will **not** be exposed as an inference feature in production.

In [ ]:
all_predictions = model.predict(X)

all_predictions = np.clip(
    all_predictions,
    a_min=0,
    a_max=None,
)

scored_customers = modeling_df[
    ["customer_unique_id"]
    + feature_columns
    + ["future_revenue"]
].copy()

scored_customers["predicted_future_revenue"] = all_predictions

scored_customers["prediction_error"] = (
    scored_customers["predicted_future_revenue"]
    - scored_customers["future_revenue"]
)

scored_customers["predicted_value_rank"] = (
    scored_customers["predicted_future_revenue"]
    .rank(method="first", ascending=False)
    .astype(int)
)

display(
    scored_customers.sort_values(
        "predicted_future_revenue",
        ascending=False,
    ).head(20)
)

## 14. Value bands for business use

The bands are percentile-based rather than hardcoded currency thresholds.

This keeps the segmentation robust when the model is retrained on another dataset or time period.

In [ ]:
scored_customers["predicted_value_band"] = pd.qcut(
    scored_customers["predicted_future_revenue"].rank(method="first"),
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High",
    ],
)

band_summary = (
    scored_customers
    .groupby("predicted_value_band", observed=False)
    .agg(
        customers=("customer_unique_id", "count"),
        average_predicted_value=(
            "predicted_future_revenue",
            "mean",
        ),
        total_predicted_value=(
            "predicted_future_revenue",
            "sum",
        ),
    )
    .reset_index()
)

display(band_summary.round(2))

## 15. Save model artifacts

The model artifact stores the estimator and the exact feature list.

This prevents production inference from accidentally using a different feature order.

In [ ]:
MODEL_PATH = MODEL_DIR / "clv_xgboost.joblib"
SCORED_PATH = OUTPUT_DIR / "customer_clv_predictions.csv"

artifact = {
    "model": model,
    "features": feature_columns,
    "target": "future_revenue",
    "observation_cutoff": cutoff_timestamp,
    "metrics": {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    },
}

joblib.dump(
    artifact,
    MODEL_PATH,
)

scored_customers.to_csv(
    SCORED_PATH,
    index=False,
)

print(f"Saved model: {MODEL_PATH}")
print(f"Saved predictions: {SCORED_PATH}")

# Final validation

The expected artifacts are:

```text
models/clv_xgboost.joblib
data/processed/customer_clv_predictions.csv
```

### Interpretation discipline

Do not claim:

> "The model predicts a customer's true lifetime value."

The defensible portfolio statement is:

> **"The model predicts future customer revenue using historical customer behavior."**

This distinction matters because the dataset has a finite historical observation window.

The next stage will add **SHAP-based customer-level explanations** to this model.